# MCP with LangGraph: Extending Agents with Model Context Protocol

**The Challenge:** You want to give your LangGraph agents access to external tools and resources—web search, databases, APIs—but managing these integrations directly in your agent code creates tight coupling and maintenance headaches. Each new capability requires code changes, and different tools have different interfaces.

**The Solution:** Model Context Protocol (MCP) provides a standardized way to expose tools and resources to AI applications. With `langchain-mcp-adapters`, you can seamlessly integrate MCP servers into your LangGraph agents, giving them access to powerful external capabilities without cluttering your agent code.

This notebook demonstrates:
1. **MCP Principles:** What MCP is and why it matters
2. **Simple Server:** Build a basic file manager MCP server
3. **LangGraph Integration:** Use MCP tools in StateGraph agents
4. **Production Example:** SQLite inventory manager with typed IO, safety, and governance
5. **Third-Party Integration:** Quick example with external MCP servers


<br>

## Best Practices

### 1. **Transport Selection**
- **stdio**: Best for local development, single-user scenarios
- **HTTP/Streamable HTTP**: Better for web servers, multi-user scenarios

### 2. **Concurrency Safety**
- Use per-call connections (not global connections) for concurrent tool calls
- Enable SQLite WAL mode for better concurrency
- Use atomic updates to prevent race conditions

### 3. **Governance**
- Enforce policies at the MCP server boundary, not in the model
- MCP servers are the governance layer for multi-agent systems

### 4. **Typed IO**
- Use Pydantic models for both input and output to ensure explicit contracts
- This prevents silent failures and makes tool behavior predictable

## Installation

First, let's install the required packages:

In [ ]:
# Install required packages
%pip install langchain-mcp-adapters langgraph langchain-core langchain-openai mcp fastmcp pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.2/416.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.8/199.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331

# Understanding MCP

**Model Context Protocol (MCP)** is an open protocol that standardizes how AI applications interact with external tools and resources. Think of it as a universal adapter that lets your agents access capabilities without knowing the implementation details.

### Key Concepts

- **MCP Servers:** Provide tools and resources (e.g., file operations, database access, web search)
- **MCP Clients:** Connect to servers and expose their capabilities to AI applications
- **Transports:** Communication methods (stdio, HTTP, SSE, streamable HTTP)
- **Tools:** Functions that agents can call (e.g., `read_file`, `search_web`)

### Why MCP Matters

- **Decoupling:** Tools live in separate servers, not in your agent code
- **Reusability:** One MCP server can serve multiple agents
- **Standardization:** Consistent interface across different tools
- **Governance:** MCP servers enforce policies and invariants outside the model

# Simple MCP Server - File Manager

Let's start with a simple file manager MCP server to understand the basics. This demonstrates the core pattern: define tools with Pydantic models, expose them via FastMCP. This simple server demonstrates:

- Typed IO with Pydantic models
- Basic tool definitions with FastMCP
- Standard MCP server pattern

In [ ]:
# Create a simple file manager MCP server
file_manager_code = '''
"""
Simple File Manager MCP Server

A minimal example showing:
- Typed IO with Pydantic models
- Basic file operations
- FastMCP server setup
"""

from pathlib import Path
from typing import Optional
from pydantic import BaseModel, Field
from mcp.server.fastmcp import FastMCP

# Pydantic models for typed IO
class ReadFileRequest(BaseModel):
    """Read a file."""
    file_path: str = Field(..., description="Path to file to read")

class WriteFileRequest(BaseModel):
    """Write content to a file."""
    file_path: str = Field(..., description="Path to file to write")
    content: str = Field(..., description="Content to write")

class ListFilesRequest(BaseModel):
    """List files in a directory."""
    directory: str = Field(..., description="Directory path")
    pattern: Optional[str] = Field(None, description="Optional glob pattern (e.g., '*.py')")

# Create MCP server
mcp = FastMCP("FileManager")

@mcp.tool()
def read_file(request: ReadFileRequest) -> str:
    """Read the contents of a file."""
    path = Path(request.file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {request.file_path}")
    return path.read_text(encoding="utf-8")

@mcp.tool()
def write_file(request: WriteFileRequest) -> dict:
    """Write content to a file. Creates file if it doesn't exist."""
    path = Path(request.file_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(request.content, encoding="utf-8")
    return {"status": "success", "file_path": str(path), "bytes_written": len(request.content)}

@mcp.tool()
def list_files(request: ListFilesRequest) -> list[str]:
    """List files in a directory."""
    dir_path = Path(request.directory)
    if not dir_path.exists():
        raise FileNotFoundError(f"Directory not found: {request.directory}")

    if request.pattern:
        files = list(dir_path.glob(request.pattern))
    else:
        files = list(dir_path.iterdir())

    return [str(f.relative_to(dir_path)) for f in files if f.is_file()]

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# Write the server file
from pathlib import Path
file_manager_path = Path("file_manager_server.py")
file_manager_path.write_text(file_manager_code)
print(f"Created {file_manager_path}")


Created file_manager_server.py


# Connect to MCP Server and Use with LangGraph

Now let's connect to our MCP server and integrate it with LangGraph. This is where the real power of MCP shines—your agent can use external tools without knowing their implementation.

## Set up your API key (or use environment variable)

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
EXA_API_KEY = os.environ.get("EXA_API_KEY", "")

## Monkey patch `OutStream.fileno`

This keeps Jupyter printing intact and simply makes fileno() stop throwing errors in Jupyter Notebooks. **Note**: You don't need this is a normal Python Script.

In [ ]:
from ipykernel.iostream import OutStream

def _safe_fileno(self):
    name = getattr(self, "name", "")
    if "stderr" in name:
        return 2
    return 1

OutStream.fileno = _safe_fileno


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.chat_models import init_chat_model

# Initialize model
try:
    model = init_chat_model("openrouter:anthropic/claude-3.5-haiku")
except:
    model = init_chat_model("openai:gpt-4o-mini")

# Create MCP client and connect to file manager server
client = MultiServerMCPClient(
    {
        "file_manager": {
            "command": "python",
            "args": [os.path.abspath("file_manager_server.py")],
            "transport": "stdio",
        }
    }
)


# Load tools from MCP server
tools = await client.get_tools()

print(f" Loaded {len(tools)} tools from MCP server:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

# Define the agent node
def call_model(state: MessagesState):
    """Call the model with tools bound."""
    response = model.bind_tools(tools).invoke(state["messages"])
    return {"messages": [response]}

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "call_model")
builder.add_conditional_edges("call_model", tools_condition)
builder.add_edge("tools", "call_model")

graph = builder.compile()
print("\n LangGraph agent created with MCP tools")

 Loaded 3 tools from MCP server:
  - read_file: Read the contents of a file.
  - write_file: Write content to a file. Creates file if it doesn't exist.
  - list_files: List files in a directory.

 LangGraph agent created with MCP tools


## Test the agent with file operations

In [ ]:

response = await graph.ainvoke({
    "messages": [("user", "Create a file called 'test.txt' with content 'Hello from MCP!', then read it back.")]
})

# Display the response
from langchain_core.messages import AIMessage
for message in response["messages"]:
    if isinstance(message, AIMessage):
        print("Agent Response:")
        print(message.content)
        if message.tool_calls:
            print("\nTool Calls:")
            for tool_call in message.tool_calls:
                print(f"  - {tool_call['name']}({tool_call['args']})")

Agent Response:


Tool Calls:
  - write_file({'request': {'file_path': 'test.txt', 'content': 'Hello from MCP!'}})
  - read_file({'request': {'file_path': 'test.txt'}})
Agent Response:
The file 'test.txt' has been created with the content 'Hello from MCP!'. When read back, the content is: **Hello from MCP!**


# Production Example - Inventory Manager

Now let's see a production-ready example: a SQLite inventory manager that demonstrates real-world patterns. This shows how MCP servers can provide persistent state, safety guarantees, and policy enforcement.

The inventory manager demonstrates the enterprise pattern:
- **Departments own systems** → MCP servers expose them
- **Agent systems consume** → Stable, typed tool contracts
- **MCP = Interop layer** → Multiple MAS integrate without code sharing
- **Governance at boundary** → Policies enforced in server, not model


In [ ]:
# Create a production-ready inventory manager MCP server
inventory_server_code = '''
"""
SQLite Inventory Manager MCP Server

Production patterns:
- Typed IO with Pydantic models (input and output)
- Safety: Atomic updates prevent negative stock
- Concurrency: Per-call connections, SQLite WAL mode
- Governance: Policy gates enforced at MCP boundary
"""

import sqlite3
import json
from pathlib import Path
from typing import Optional, List
from contextlib import contextmanager
from pydantic import BaseModel, Field, field_validator
from mcp.server.fastmcp import FastMCP

# ============================================================================
# Pydantic Models
# ============================================================================

class ItemCreate(BaseModel):
    """Create a new inventory item."""
    name: str = Field(..., min_length=1, max_length=200)
    initial_stock: int = Field(..., ge=0)
    unit_price: float = Field(..., ge=0)
    category: Optional[str] = None

class StockAdjustment(BaseModel):
    """Adjust stock quantity."""
    item_id: int = Field(..., gt=0)
    quantity: int = Field(..., description="Positive adds, negative removes")
    reason: str = Field(..., min_length=1)
    approval_token: Optional[str] = None

    @field_validator("quantity")
    @classmethod
    def quantity_not_zero(cls, v):
        if v == 0:
            raise ValueError("Quantity must be non-zero")
        return v

class ItemQuery(BaseModel):
    """Query items."""
    item_id: Optional[int] = None
    category: Optional[str] = None
    low_stock_threshold: Optional[int] = None

class ItemOut(BaseModel):
    """Item output model."""
    id: int
    name: str
    stock: int
    unit_price: float
    category: Optional[str]

# ============================================================================
# Database Setup
# ============================================================================

DB_PATH = Path("inventory.db")

def init_database():
    """Initialize database schema."""
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("PRAGMA foreign_keys=ON")

    conn.execute("""
        CREATE TABLE IF NOT EXISTS items (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
            unit_price REAL NOT NULL CHECK(unit_price >= 0),
            category TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS audit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            item_id INTEGER,
            event_type TEXT NOT NULL,
            event_data TEXT NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (item_id) REFERENCES items(id)
        )
    """)

    conn.commit()
    conn.close()

init_database()

def connect():
    """Create a new connection for each tool call (safe for concurrency)."""
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("PRAGMA foreign_keys=ON")
    return conn

@contextmanager
def transaction():
    """Context manager for transactions with per-call connections."""
    conn = connect()
    try:
        conn.execute("BEGIN")
        yield conn
        conn.execute("COMMIT")
    except Exception:
        conn.execute("ROLLBACK")
        raise
    finally:
        conn.close()

def log_event(conn, item_id: Optional[int], event_type: str, event_data: dict):
    """Append event to audit log using active connection."""
    conn.execute(
        "INSERT INTO audit_log (item_id, event_type, event_data) VALUES (?, ?, ?)",
        (item_id, event_type, json.dumps(event_data))
    )

# ============================================================================
# Policy Enforcement
# ============================================================================

def enforce_policy(adjustment: StockAdjustment):
    """Enforce business policies - MCP servers are the governance boundary."""
    # Policy: Large reductions require approval
    if adjustment.quantity < -100:
        if not adjustment.approval_token or adjustment.approval_token != "APPROVED_BY_MANAGER":
            raise ValueError(
                f"Large stock reduction ({abs(adjustment.quantity)} units) requires approval_token."
            )

    # Policy: Negative adjustments must reference an order
    if adjustment.quantity < 0:
        if not adjustment.reason.startswith("ORDER:"):
            raise ValueError(
                "Negative adjustments must include order reference. Format: 'ORDER: <id> - <desc>'"
            )

# ============================================================================
# MCP Server
# ============================================================================

mcp = FastMCP("InventoryManager")

@mcp.tool()
def create_item(item: ItemCreate) -> ItemOut:
    """Create a new inventory item."""
    with transaction() as conn:
        cursor = conn.execute(
            "INSERT INTO items (name, stock, unit_price, category) VALUES (?, ?, ?, ?)",
            (item.name, item.initial_stock, item.unit_price, item.category)
        )
        item_id = cursor.lastrowid

        log_event(conn, item_id, "item_created", {
            "name": item.name,
            "initial_stock": item.initial_stock
        })

        row = conn.execute(
            "SELECT id, name, stock, unit_price, category FROM items WHERE id = ?",
            (item_id,)
        ).fetchone()

        return ItemOut(
            id=row["id"],
            name=row["name"],
            stock=row["stock"],
            unit_price=row["unit_price"],
            category=row["category"]
        )

@mcp.tool()
def adjust_stock(adjustment: StockAdjustment) -> ItemOut:
    """Adjust stock quantity. Enforces no negative stock and policy gates."""
    # Enforce policy gates (governance boundary)
    enforce_policy(adjustment)

    with transaction() as conn:
        # Atomic update: only succeeds if stock won't go negative
        cur = conn.execute(
            "UPDATE items SET stock = stock + ? WHERE id = ? AND stock + ? >= 0",
            (adjustment.quantity, adjustment.item_id, adjustment.quantity)
        )

        if cur.rowcount == 0:
            exists = conn.execute("SELECT 1 FROM items WHERE id = ?", (adjustment.item_id,)).fetchone()
            if not exists:
                raise ValueError(f"Item {adjustment.item_id} not found")

            current = conn.execute("SELECT stock FROM items WHERE id = ?", (adjustment.item_id,)).fetchone()
            current_stock = current["stock"] if current else 0
            raise ValueError(
                f"Insufficient stock. Current: {current_stock}, requested: {adjustment.quantity}"
            )

        updated = conn.execute("SELECT stock FROM items WHERE id = ?", (adjustment.item_id,)).fetchone()
        new_stock = updated["stock"]

        log_event(conn, adjustment.item_id, "stock_adjusted", {
            "adjustment": adjustment.quantity,
            "new_stock": new_stock,
            "reason": adjustment.reason
        })

        row = conn.execute(
            "SELECT id, name, stock, unit_price, category FROM items WHERE id = ?",
            (adjustment.item_id,)
        ).fetchone()

        return ItemOut(
            id=row["id"],
            name=row["name"],
            stock=row["stock"],
            unit_price=row["unit_price"],
            category=row["category"]
        )

@mcp.tool()
def list_items(query: ItemQuery) -> List[ItemOut]:
    """Query inventory items."""
    conn = connect()
    try:
        conditions = []
        params = []

        if query.item_id:
            conditions.append("id = ?")
            params.append(query.item_id)
        if query.category:
            conditions.append("category = ?")
            params.append(query.category)
        if query.low_stock_threshold is not None:
            conditions.append("stock <= ?")
            params.append(query.low_stock_threshold)

        where = "WHERE " + " AND ".join(conditions) if conditions else ""

        rows = conn.execute(
            f"SELECT id, name, stock, unit_price, category FROM items {where} ORDER BY name",
            params
        ).fetchall()

        return [
            ItemOut(
                id=row["id"],
                name=row["name"],
                stock=row["stock"],
                unit_price=row["unit_price"],
                category=row["category"]
            )
            for row in rows
        ]
    finally:
        conn.close()

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# Write the server file
inventory_path = Path("inventory_server.py")
inventory_path.write_text(inventory_server_code)
print(f"Created {inventory_path}")
print("\nKey patterns demonstrated:")
print("  • Typed IO: Pydantic models for contracts")
print("  • Safety: Atomic updates prevent race conditions")
print("  • Concurrency: Per-call connections, WAL mode")
print("  • Governance: Policy gates enforced at server boundary")

Created inventory_server.py

Key patterns demonstrated:
  • Typed IO: Pydantic models for contracts
  • Safety: Atomic updates prevent race conditions
  • Concurrency: Per-call connections, WAL mode
  • Governance: Policy gates enforced at server boundary


In [ ]:
# Create agent with inventory manager
inventory_client = MultiServerMCPClient(
    {
        "inventory": {
            "command": "python",
            "args": [os.path.abspath("inventory_server.py")],
            "transport": "stdio",
        }
    }
)

inventory_tools = await inventory_client.get_tools()

def call_model_inventory(state: MessagesState):
    response = model.bind_tools(inventory_tools).invoke(state["messages"])
    return {"messages": [response]}

builder_inv = StateGraph(MessagesState)
builder_inv.add_node("call_model", call_model_inventory)
builder_inv.add_node("tools", ToolNode(inventory_tools))
builder_inv.add_edge(START, "call_model")
builder_inv.add_conditional_edges("call_model", tools_condition)
builder_inv.add_edge("tools", "call_model")

inventory_graph = builder_inv.compile()

# Demo: Create item, adjust stock, test safety and policy gates
print("=== Inventory Manager Demo ===\n")

# 1. Create an item
print("1. Creating item...")
response1 = await inventory_graph.ainvoke({
    "messages": [("user", "Create a new item: 'Laptop Pro' with initial stock 10, price $1299.99, category 'Electronics'")]
})
print("   ✅ Item created\n")

# 2. Adjust stock (positive)
print("2. Adding stock...")
response2 = await inventory_graph.ainvoke({
    "messages": [("user", "Add 5 units to item ID 1, reason: 'Received shipment'")]
})
print("   ✅ Stock increased\n")

# 3. Test safety: Try to remove more than available
print("3. Testing safety invariant (prevent negative stock)...")
try:
    await inventory_graph.ainvoke({
        "messages": [("user", "Remove 1000 units from item ID 1, reason: 'ORDER: TEST-001 - Test order'")]
    })
    print("   ⚠️  Safety check failed!")
except Exception as e:
    print(f"   ✅ Safety check passed: {str(e)[:60]}...\n")

# 4. Test policy: Large reduction without approval
print("4. Testing policy gate (large reduction requires approval)...")
try:
    await inventory_graph.ainvoke({
        "messages": [("user", "Reduce stock by 150 units for item ID 1, reason: 'ORDER: BULK-001 - Bulk order'")]
    })
    print("   ⚠️  Policy check failed!")
except Exception as e:
    print(f"   ✅ Policy gate enforced: {str(e)[:60]}...\n")

print("=== Key Insights ===")
print("  • MCP servers enforce safety invariants (no negative stock)")
print("  • Policy gates are enforced at the server boundary, not in the model")
print("  • Multiple agents can safely share the same database through MCP")

=== Inventory Manager Demo ===

1. Creating item...
   ✅ Item created

2. Adding stock...
   ✅ Stock increased

3. Testing safety invariant (prevent negative stock)...
   ✅ Safety check passed: Error executing tool adjust_stock: Large stock reduction (10...

4. Testing policy gate (large reduction requires approval)...
   ✅ Policy gate enforced: Error executing tool adjust_stock: Large stock reduction (15...

=== Key Insights ===
  • MCP servers enforce safety invariants (no negative stock)
  • Policy gates are enforced at the server boundary, not in the model
  • Multiple agents can safely share the same database through MCP


# Third-Party MCP Servers

Many services provide MCP servers for easy integration. Here's a quick example using Exa's MCP server for web search:

In [ ]:
# Example: Combine inventory manager with Exa search
EXA_MCP_URL = "https://mcp.exa.ai/mcp"


if EXA_API_KEY:
    # Combine multiple MCP servers
    combined_client = MultiServerMCPClient(
        {
            "inventory": {
                "command": "python",
                "args": [os.path.abspath("inventory_server.py")],
                "transport": "stdio",
            },
            "exa": {
                "transport": "http",
                "url": EXA_MCP_URL,
                "headers": {"Authorization": f"Bearer {EXA_API_KEY}"},
            }
        }
    )

    combined_tools = await combined_client.get_tools()
    print(f"✅ Loaded {len(combined_tools)} tools from multiple MCP servers")
    print("   You can now use both inventory management and web search in the same agent")
else:
    print("⚠️  Set EXA_API_KEY to test third-party MCP integration")
    print("   Example: os.environ['EXA_API_KEY'] = 'your-key-here'")

✅ Loaded 5 tools from multiple MCP servers
   You can now use both inventory management and web search in the same agent
